In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

rfm = pd.read_csv('../data/processed/rfm_segmented.csv')
rfm.head()

,CustomerID,FirstName,LastName,TerritoryName,LastOrderDate,Frequency,Monetary,Recency,FirstOrderDate,Tenure,AvgOrderValue,AvgPurchaseGap,Churned,R_Score,F_Score,M_Score,RFM_Score,Segment
0,11000,Jon,Yang,Australia,2013-10-03,3,9115.1341,270,2011-06-21,1105,3038.378033,417.5,1,2,5,5,255,At Risk
1,11001,Eugene,Huang,Australia,2014-05-12,3,7054.1875,49,2011-06-17,1109,2351.395833,530.0,0,5,5,5,555,Champions
2,11002,Ruben,Torres,Australia,2013-07-26,3,8966.0143,339,2011-06-09,1117,2988.671433,389.0,1,1,5,5,155,At Risk
3,11003,Christy,Zhu,Australia,2013-10-10,3,8993.9155,263,2011-05-31,1126,2997.971833,431.5,1,2,5,5,255,At Risk
4,11004,Elizabeth,Johnson,Australia,2013-10-01,3,9056.5911,272,2011-06-25,1101,3018.863700,414.5,1,2,5,5,255,At Risk


In [37]:
rfm_model = pd.get_dummies(rfm, columns=['TerritoryName'], drop_first=True)

feature_cols = ['Frequency', 'Monetary', 'AvgOrderValue', 'AvgPurchaseGap'] + \
                [col for col in rfm_model.columns if col.startswith('TerritoryName_')]

X = rfm_model[feature_cols]
y = rfm_model['Churned']

X.head()


,Frequency,Monetary,AvgOrderValue,AvgPurchaseGap,TerritoryName_Canada,TerritoryName_Central,TerritoryName_France,TerritoryName_Germany,TerritoryName_Northeast,TerritoryName_Northwest,TerritoryName_Southeast,TerritoryName_Southwest,TerritoryName_United Kingdom
0,3,9115.1341,3038.378033,417.5,False,False,False,False,False,False,False,False,False
1,3,7054.1875,2351.395833,530.0,False,False,False,False,False,False,False,False,False
2,3,8966.0143,2988.671433,389.0,False,False,False,False,False,False,False,False,False
3,3,8993.9155,2997.971833,431.5,False,False,False,False,False,False,False,False,False
4,3,9056.5911,3018.863700,414.5,False,False,False,False,False,False,False,False,False


In [38]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()

,Frequency,Monetary,AvgOrderValue,AvgPurchaseGap,TerritoryName_Canada,TerritoryName_Central,TerritoryName_France,TerritoryName_Germany,TerritoryName_Northeast,TerritoryName_Northwest,TerritoryName_Southeast,TerritoryName_Southwest,TerritoryName_United Kingdom
0,0.929472,0.061031,0.225409,1.203621,-0.310076,-0.060183,-0.326717,-0.32357,-0.054683,-0.467407,-0.069155,-0.560053,-0.337108
1,0.929472,0.013929,0.111569,1.686402,-0.310076,-0.060183,-0.326717,-0.32357,-0.054683,-0.467407,-0.069155,-0.560053,-0.337108
2,0.929472,0.057623,0.217172,1.081316,-0.310076,-0.060183,-0.326717,-0.32357,-0.054683,-0.467407,-0.069155,-0.560053,-0.337108
3,0.929472,0.058260,0.218713,1.263700,-0.310076,-0.060183,-0.326717,-0.32357,-0.054683,-0.467407,-0.069155,-0.560053,-0.337108
4,0.929472,0.059693,0.222175,1.190746,-0.310076,-0.060183,-0.326717,-0.32357,-0.054683,-0.467407,-0.069155,-0.560053,-0.337108


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(15295, 13) (3824, 13)


In [40]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.5444560669456067
Precision: 0.5026634382566586
Recall: 0.5921277809469481
F1 Score: 0.5437401781037192


In [42]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n", classification_report(y_test, y_pred))

[[1044 1027]
 [ 715 1038]]

               precision    recall  f1-score   support

           0       0.59      0.50      0.55      2071
           1       0.50      0.59      0.54      1753

    accuracy                           0.54      3824
   macro avg       0.55      0.55      0.54      3824
weighted avg       0.55      0.54      0.54      3824



In [43]:
correlation = rfm['Tenure'].corr(rfm['Recency'])
print("Correlation between Tenure and Recency:", correlation)

one_time_buyers = rfm[rfm['Frequency'] == 1]
print("One-time buyers where Tenure == Recency:", (one_time_buyers['Tenure'] == one_time_buyers['Recency']).sum(), "out of", len(one_time_buyers))

Correlation between Tenure and Recency: 0.38045931505435565
One-time buyers where Tenure == Recency: 11649 out of 11649


In [44]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42, n_estimators=200)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))

Random Forest Results:
Accuracy: 0.6582112970711297
Precision: 0.6280137772675086
Recall: 0.6240730176839704
F1 Score: 0.6260371959942775


In [45]:
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

importances

,Feature,Importance
2,AvgOrderValue,0.333458
1,Monetary,0.323592
3,AvgPurchaseGap,0.244261
0,Frequency,0.048000
12,TerritoryName_United Kingdom,0.009138
11,TerritoryName_Southwest,0.008075
7,TerritoryName_Germany,0.007948
9,TerritoryName_Northwest,0.007574
6,TerritoryName_France,0.007288
4,TerritoryName_Canada,0.007077
